<a href="https://colab.research.google.com/github/aarohigorle2005/Data_Science_Lab/blob/main/Experiment8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Experiment 8: Movie Recommendation Using Rating Prediction (MovieLens Dataset)

import os
import requests
import zipfile
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report

# 1. Download MovieLens Dataset
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

if not os.path.exists("ml-latest-small"):
    print("Downloading dataset...")
    r = requests.get(url)
    with open("movielens.zip", "wb") as f:
        f.write(r.content)

    print("Extracting dataset...")
    with zipfile.ZipFile("movielens.zip", 'r') as zip_ref:
        zip_ref.extractall()

# 2. Load Dataset
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")

# Merge datasets
data = pd.merge(ratings, movies, on="movieId")

print("Data Sample:\n", data.head())

# 3. Preprocessing
# Convert genres to numeric
data['genres'] = data['genres'].astype('category').cat.codes

X = data[['userId', 'movieId', 'genres']]

# 4. REGRESSION
y_reg = data['rating']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
reg_model.fit(X_train_r, y_train_r)

y_pred_r = reg_model.predict(X_test_r)

rmse = np.sqrt(mean_squared_error(y_test_r, y_pred_r))
print("\nRegression RMSE:", rmse)

# 5. CLASSIFICATION
data['rating_class'] = np.where(data['rating'] >= 3.5, 1, 0)
y_cls = data['rating_class']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42
)

clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
clf_model.fit(X_train_c, y_train_c)

y_pred_c = clf_model.predict(X_test_c)

print("\nClassification Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("\nClassification Report:\n", classification_report(y_test_c, y_pred_c))

# 6. Sample Prediction
sample = X.iloc[[0]]

print("\nSample Input:\n", sample)
print("Predicted Rating:", reg_model.predict(sample)[0])
print("Predicted Class (1=High, 0=Low):", clf_model.predict(sample)[0])

Data Sample:
    userId  movieId  rating  timestamp                        title  \
0       1        1     4.0  964982703             Toy Story (1995)   
1       1        3     4.0  964981247      Grumpier Old Men (1995)   
2       1        6     4.0  964982224                  Heat (1995)   
3       1       47     5.0  964983815  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0  964982931   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  

Regression RMSE: 0.9652945690141774

Classification Accuracy: 0.6586671955573186

Classification Report:
               precision    recall  f1-score   support

           0       0.57      0.51      0.54      7829
           1       0.71      0.75      0.73     12339

